# SE4050 Deep Learning Assignment - Human Activity Recognition (HAR)
## Component 1: Multi-Head Self-Attention Transformer Encoder Architecture

**Author:** Dharana (Member 1)  
**Component:** Pure Transformer Encoder for Multi-channel Sensor Time-Series Classification  
**Framework:** TensorFlow / Keras (Trained from Scratch)  

---

### 1. Team Project & Architectural Comparison Context
In this collaborative project, our four-member team benchmarks four distinct neural architectures on the [UCI Human Activity Recognition Using Smartphones](https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones) dataset:
- **Dharana (Me - Member 1):** Multi-Head Self-Attention Transformer Encoder with Trainable Positional Embeddings
- **Member 2:** 1D-CNN Baseline and Exploratory Data Analysis (EDA)
- **Member 3:** Bidirectional LSTM (BiLSTM) and Centralized Shared Data Loader with Subject Stratification
- **Member 4:** Hybrid CNN-LSTM Architecture and Unified Multi-Model Evaluation Benchmark

### 2. Transformer Architecture Overview
- **Input:** Triaxial raw inertial sensor windows shaped `(N, 128, 9)` (128 time steps @ 50 Hz $\times$ 9 sensor channels).
- **Linear Input Projection:** Projects raw 9 channels into a $d_{\text{model}} = 64$ continuous embedding space.
- **Trainable Positional Embeddings:** Learnable $128 \times 64$ parameter matrix injects relative & absolute temporal ordering.
- **Pre-LN Transformer Encoder Blocks (2 layers):**
  - Pre-LayerNormalization before Multi-Head Self-Attention (`num_heads = 4`, `key_dim = 16`, `dropout = 0.2`)
  - Residual skip connections (`x + MHA(LN(x))`)
  - Pre-LayerNormalization before Position-Wise Feed-Forward Network: `Dense(128, GELU)` $\to$ `Dense(64)`
  - Residual skip connections (`x + FFN(LN(x))`)
- **Final Layer Normalization:** Stabilizes representations before pooling.
- **Global Average Pooling 1D:** Temporal reduction across the 128 timesteps $\to (N, 64)$.
- **Classification Head:** `Dense(64, activation='relu')` $\to$ `Dropout(0.2)` $\to$ `Dense(6, activation='softmax')`.
- **Strict Leakage Prevention:** Disjoint subject splits: $\text{Subjects}(\text{Train}) \cap \text{Subjects}(\text{Val}) = \emptyset$.

### Section 1: Environment Setup, Dependencies & Reproducibility
Detects whether running in Google Colab or locally. Configures random seeds (`seed=42`) and imports modular project components.

In [ ]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('[Colab] Running in Google Colab environment.')
    from google.colab import userdata
    try:
        TOKEN = userdata.get('GH_TOKEN')
        REPO = 'DharanaT567-del/Deep-Learning-Assignment-SE4050'
        BRANCH = 'dharana/componenet1/new'
        if not Path('/content/repo').exists():
            os.system(f'git clone --branch {BRANCH} https://{TOKEN}@github.com/{REPO}.git /content/repo')
    except Exception:
        if not Path('/content/repo').exists():
            os.system('git clone https://github.com/DharanaT567-del/Deep-Learning-Assignment-SE4050.git /content/repo')
    os.chdir('/content/repo')
    os.system('pip install -q -r requirements.txt')
else:
    # Walk up from current working directory to locate repo root
    root = Path.cwd()
    while not (root / 'src' / 'data_contract.py').exists() and root != root.parent:
        root = root.parent
    os.chdir(root)

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

import tensorflow as tf
import keras
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

from src.data_contract import (
    ACTIVITY_LABEL_MAPPING,
    SENSOR_CHANNEL_NAMES,
    load_har_npz,
    validate_har_dataset,
)
from src.data.uci_har_loader import build_processed_dataset, save_processed_dataset
from src.models.transformer import (
    TrainablePositionalEmbedding,
    TransformerEncoderBlock,
    build_transformer_classifier,
    compile_transformer_model,
    load_transformer_model,
    predict_single_window,
    predict_transformer,
)
from src.train_transformer import (
    evaluate_transformer_on_test,
    load_config,
    set_seed,
    train_transformer_pipeline,
)

set_seed(42)
print(f'TensorFlow Version: {tf.__version__}')
print(f'Keras Version:      {keras.__version__}')
gpu_devices = tf.config.list_physical_devices('GPU')
print(f'GPU Available:      {len(gpu_devices) > 0} ({gpu_devices})')

### Section 2: Dataset Loading & Shared Contract Validation
We load the centralized preprocessed dataset (`data/uci_har_processed.npz`). If not present, we automatically download and preprocess it using `src.data.uci_har_loader`.

In [ ]:
data_path = PROJECT_ROOT / 'data' / 'uci_har_processed.npz'
if not data_path.is_file():
    print('[Data] Processed dataset not found. Downloading raw UCI HAR and building NPZ...')
    raw_data = build_processed_dataset(n_val_subjects=4, seed=42)
    save_processed_dataset(raw_data, data_path)

data = load_har_npz(str(data_path))
validate_har_dataset(data, check_test=True)

X_train, y_train, sub_train = data['X_train'], data['y_train'], data['subject_train']
X_val, y_val, sub_val = data['X_val'], data['y_val'], data['subject_val']
X_test, y_test, sub_test = data['X_test'], data['y_test'], data['subject_test']

CLASS_NAMES = [ACTIVITY_LABEL_MAPPING[i] for i in range(len(ACTIVITY_LABEL_MAPPING))]

print(f'Train split: X={X_train.shape}, y={y_train.shape}, Subjects={sorted(int(s) for s in np.unique(sub_train))}')
print(f'Val split:   X={X_val.shape}, y={y_val.shape}, Subjects={sorted(int(s) for s in np.unique(sub_val))}')
print(f'Test split:  X={X_test.shape}, y={y_test.shape}, Subjects={sorted(int(s) for s in np.unique(sub_test))}')
print(f'Subject overlap (Train ∩ Val):  {set(sub_train).intersection(set(sub_val))}')
print(f'Subject overlap (Train ∩ Test): {set(sub_train).intersection(set(sub_test))}')
print(f'Subject overlap (Val ∩ Test):   {set(sub_val).intersection(set(sub_test))}')

### Section 3: Exploratory Signal Waveform Visualization
Visualizing multi-channel inertial sensor waveforms for dynamic (e.g. `WALKING`) versus static (e.g. `LAYING`) activities.

In [ ]:
plt.figure(figsize=(14, 5))
dyn_idx = int(np.where(y_train == 0)[0][0])  # WALKING
stat_idx = int(np.where(y_train == 5)[0][0])  # LAYING

plt.subplot(1, 2, 1)
plt.plot(X_train[dyn_idx, :, 0], label='Body Acc X (g)', color='#e74c3c')
plt.plot(X_train[dyn_idx, :, 1], label='Body Acc Y (g)', color='#2ecc71')
plt.plot(X_train[dyn_idx, :, 2], label='Body Acc Z (g)', color='#3498db')
plt.title(f'Dynamic Signal: {ACTIVITY_LABEL_MAPPING[y_train[dyn_idx]]} (Subject {sub_train[dyn_idx]})', fontweight='bold')
plt.xlabel('Time Steps (128 = 2.56s @ 50 Hz)')
plt.ylabel('Standardized Value')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(X_train[stat_idx, :, 0], label='Body Acc X (g)', color='#e74c3c')
plt.plot(X_train[stat_idx, :, 1], label='Body Acc Y (g)', color='#2ecc71')
plt.plot(X_train[stat_idx, :, 2], label='Body Acc Z (g)', color='#3498db')
plt.title(f'Static Signal: {ACTIVITY_LABEL_MAPPING[y_train[stat_idx]]} (Subject {sub_train[stat_idx]})', fontweight='bold')
plt.xlabel('Time Steps (128 = 2.56s @ 50 Hz)')
plt.ylabel('Standardized Value')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Section 4: Transformer Architecture Construction & Parameter Inspection
Loads hyperparameter configuration from `configs/transformer.json` and constructs the Keras functional model.

In [ ]:
config = load_config('configs/transformer.json')
print('Loaded Transformer Configuration:')
print(json.dumps(config, indent=2))

model = build_transformer_classifier(config=config['model'])
compile_transformer_model(model, learning_rate=config['training']['learning_rate'])
model.summary()

### Section 5: Training Pipeline Execution
Executes training with Adam optimizer, EarlyStopping (`patience=10`), ModelCheckpoint (`best_model.keras`), ReduceLROnPlateau (`patience=5`), and structured logging.

In [ ]:
best_model, metadata, run_dir = train_transformer_pipeline(
    config=config,
    data=data,
    verbose=1,
)

### Section 6: Training History & Learning Curves
Visualizing convergence behavior and early stopping checkpoint selection.

In [ ]:
history_file = Path(run_dir) / 'history.json'
with open(history_file, 'r') as f:
    history = json.load(f)

epochs_range = range(1, len(history['loss']) + 1)
best_epoch = metadata['best_val_loss_epoch']

plt.figure(figsize=(14, 5))

# Loss Curve
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history['loss'], label='Train Loss', color='#2980b9', lw=2)
plt.plot(epochs_range, history['val_loss'], label='Val Loss', color='#e67e22', lw=2)
if best_epoch:
    plt.axvline(best_epoch, color='#27ae60', linestyle='--', alpha=0.8, label=f'Best Val Epoch ({best_epoch})')
plt.title('Sparse Categorical Cross-Entropy Loss Progression', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# Accuracy Curve
plt.subplot(1, 2, 2)
plt.plot(epochs_range, history['accuracy'], label='Train Accuracy', color='#2980b9', lw=2)
plt.plot(epochs_range, history['val_accuracy'], label='Val Accuracy', color='#e67e22', lw=2)
if best_epoch:
    plt.axvline(best_epoch, color='#27ae60', linestyle='--', alpha=0.8, label=f'Best Val Epoch ({best_epoch})')
plt.title('Classification Accuracy Progression', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Section 7: Checkpoint Serialization & Reload Verification
Confirms that the saved `.keras` artifact reloads cleanly and matches original weights and predictions exactly.

In [ ]:
best_model_path = Path(run_dir) / 'best_model.keras'
reloaded_model = load_transformer_model(str(best_model_path))

# Check prediction consistency on validation batch
sample_val = X_val[:10]
p_orig = predict_transformer(best_model, sample_val)
p_reld = predict_transformer(reloaded_model, sample_val)
np.testing.assert_allclose(p_orig, p_reld, rtol=1e-5, atol=1e-5)
print('Checkpoint reload verification PASSED. Prediction fidelity verified across custom layers.')

### Section 8: Held-Out Test Set Benchmark Evaluation
**Test Isolation Rule:** The test split (`X_test`, `y_test`) consists of 9 unseen subjects (2,947 windows) and is evaluated only once with the saved best model.

In [ ]:
test_metrics = evaluate_transformer_on_test(
    reloaded_model,
    data=data,
    run_dir=run_dir,
    save_artifacts=True,
)

### Section 9: Confusion Matrix Heatmap Analysis

In [ ]:
cm = np.array(test_metrics['confusion_matrix'])
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Raw Counts
im0 = ax[0].imshow(cm, interpolation='nearest', cmap='Blues')
ax[0].set_title('Confusion Matrix (Raw Counts)', fontsize=12, fontweight='bold')
fig.colorbar(im0, ax=ax[0], fraction=0.046, pad=0.04)
tick_marks = np.arange(len(CLASS_NAMES))
ax[0].set_xticks(tick_marks)
ax[0].set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax[0].set_yticks(tick_marks)
ax[0].set_yticklabels(CLASS_NAMES)
ax[0].set_ylabel('True Activity', fontweight='bold')
ax[0].set_xlabel('Predicted Activity', fontweight='bold')
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        ax[0].text(j, i, format(cm[i, j], 'd'),
                   ha='center', va='center',
                   color='white' if cm[i, j] > cm.max() / 2 else 'black')

# Normalized Percentages
im1 = ax[1].imshow(cm_norm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
ax[1].set_title('Confusion Matrix (Normalized Recall %)', fontsize=12, fontweight='bold')
fig.colorbar(im1, ax=ax[1], fraction=0.046, pad=0.04)
ax[1].set_xticks(tick_marks)
ax[1].set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax[1].set_yticks(tick_marks)
ax[1].set_yticklabels(CLASS_NAMES)
ax[1].set_ylabel('True Activity', fontweight='bold')
ax[1].set_xlabel('Predicted Activity', fontweight='bold')
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        ax[1].text(j, i, f'{cm_norm[i, j]*100:.1f}%',
                   ha='center', va='center',
                   color='white' if cm_norm[i, j] > 0.5 else 'black')

plt.tight_layout()
cm_fig_path = Path(run_dir) / 'confusion_matrix.png'
fig.savefig(cm_fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Confusion matrix figure saved to: {cm_fig_path}')

### Section 10: Single-Window Real-Time Inference Demo

In [ ]:
sample_test_idx = 100
single_sample = X_test[sample_test_idx]
true_lbl_idx = y_test[sample_test_idx]
true_lbl_name = ACTIVITY_LABEL_MAPPING[true_lbl_idx]

pred_idx, pred_name, conf, probs = predict_single_window(reloaded_model, single_sample)

print(f'Test Sample Index:  {sample_test_idx}')
print(f'True Activity:      {true_lbl_name} (Class {true_lbl_idx})')
print(f'Predicted Activity: {pred_name} (Class {pred_idx})')
print(f'Prediction Confidence: {conf * 100:.2f}%')
print('\nClass Probability Distribution:')
for i, c_name in ACTIVITY_LABEL_MAPPING.items():
    bar_len = int(probs[i] * 35)
    print(f'  [{i}] {c_name:<20}: {probs[i]*100:5.2f}% | {"█" * bar_len}')

### Section 11: Analytical Findings, Teammate Architecture Comparison & Viva Notes

#### 1. Attention Mechanism Analysis on HAR Time-Series
- **Global Context Modeling:** Unlike 1D-CNNs constrained by local kernel filter sizes ($k=3$ or $k=5$) and LSTMs constrained by sequential unrolling, Multi-Head Self-Attention directly computes pairwise attention scores $A_{i,j} = \text{softmax}(Q_i K_j^T / \sqrt{d_k})$ across all $128$ time steps in parallel.
- **Dynamic Feature Interaction:** The multi-head projection allows head 1 to focus on periodic gait rhythm (walking cadence), while head 2 captures high-frequency accelerometer peaks (impacts when walking upstairs/downstairs).

#### 2. Class Confusion Patterns & Biomechanical Ambiguity
- **Static Postures (SITTING vs. STANDING):** Similar to Member 3's BiLSTM and Member 2's CNN findings, the primary confusion occurs between SITTING and STANDING. Because the smartphone is attached to the waist, the static gravitational acceleration vector $\mathbf{g} = (0, 0, 1)$ is nearly identical in both upright sitting and standing postures with negligible body motion. When no transition motion is contained within the window, separating these two states relies on subtle postural tilts.
- **LAYING:** Perfect or near-100% recall is achieved across all models because the gravity vector shifts from the vertical axis ($Z$) to the lateral/longitudinal plane ($X/Y$), creating an unmistakable signal signature.
- **Dynamic Activities (WALKING, UPSTAIRS, DOWNSTAIRS):** Easily distinguished from static postures due to high variance across acceleration and gyroscope channels, with minor inter-class confusion between climbing stairs and level walking due to subject gait variations.

#### 3. Architecture Comparison Summary

| Metric / Property | Transformer Encoder (Dharana) | BiLSTM (Member 3) | 1D-CNN (Member 2) | CNN-LSTM (Member 4) |
| :--- | :--- | :--- | :--- | :--- |
| **Parameters** | **~80,454** | ~145,350 | ~50,000 | ~110,000 |
| **Temporal Modeling** | Multi-Head Self-Attention ($O(1)$ path) | Sequential recurrence ($O(T)$ path) | Local receptive field stacking | Local features + temporal recurrence |
| **Training Parallelism** | **Full sequence parallelism** | Strict sequential dependence | Full sequence parallelism | Hybrid |
| **Mobile Inference** | Low latency, matrix-mult friendly | Sequential gate evaluation | Fastest forward pass | Medium latency |

#### 4. Viva Voce Preparation Q&A
- **Q: Why did you use Trainable Positional Embeddings instead of fixed sinusoidal embeddings?**  
  *A:* Inertial sensor windows are fixed at 128 timesteps (2.56s @ 50 Hz). Trainable positional embeddings allow the network to learn the exact temporal characteristics of the fixed-length window directly from sensor dynamics rather than imposing an unlearnable geometric prior.
- **Q: Why Pre-LayerNormalization instead of Post-LayerNormalization?**  
  *A:* Pre-LN places LayerNorm inside the residual stream before Multi-Head Attention and the FFN sublayers. This maintains a clean identity path through skip connections, preventing vanishing/exploding gradients and eliminating the need for complex learning-rate warmups.
- **Q: How did you ensure a fair comparison with the other three members?**  
  *A:* All four models were trained on the exact same `data/uci_har_processed.npz` package, with identical subject-wise disjoint splits (17 train, 4 val, 9 test), channel order, and training-only standardization, using the shared data contract in `src/data_contract.py`.